In [1]:
# import pandas as pd

# def load_features_from_csv(input_file):
#     # Memuat file CSV
#     data = pd.read_csv(input_file)

#     # Memisahkan fitur (kolom selain label) dan label
#     features = data.drop(columns=['label']).values
#     labels = data['label'].values

#     return features, labels

# # Memuat fitur dan label dari file CSV
# train_hog_features, hog_train_labels = load_features_from_csv("hog_train_features.csv")
# test_hog_features, hog_test_labels = load_features_from_csv("hog_test_features.csv")
# train_landamrk_features, landmark_train_labels = load_features_from_csv("train_landmark_features.csv")
# test_landmark_features, landmark_test_labels = load_features_from_csv("test_landmark_features.csv")

# # Pastikan fitur dan label dimuat dengan benar
# print("Train Hog Features Shape:", train_hog_features.shape)
# print("Test Hog Features Shape:", test_hog_features.shape)
# print("Train Landmark Features Shape:", train_landamrk_features.shape)
# print("Test Landmark Features Shape:", test_landmark_features.shape)


In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def load_features_from_csv(input_file):
    # Memuat file CSV
    data = pd.read_csv(input_file)

    # Memisahkan fitur (kolom selain label) dan label
    features = data.drop(columns=['label']).values
    labels = data['label'].values

    return features, labels

# Memuat fitur dan label dari file CSV
train_hog_features, hog_train_labels = load_features_from_csv("hog_train_features.csv")
test_hog_features, hog_test_labels = load_features_from_csv("hog_test_features.csv")
train_landmark_features, landmark_train_labels = load_features_from_csv("train_landmark_features.csv")
test_landmark_features, landmark_test_labels = load_features_from_csv("test_landmark_features.csv")



# Standarisasi fitur HOG
scaler = StandardScaler()
train_hog_features_scaled = scaler.fit_transform(train_hog_features)
test_hog_features_scaled = scaler.transform(test_hog_features)

# Inisialisasi PCA untuk menghitung variansi kumulatif
pca_temp = PCA()
pca_temp.fit(train_hog_features_scaled)

# Hitung variansi kumulatif
cumulative_variance = np.cumsum(pca_temp.explained_variance_ratio_)

# Cari jumlah komponen untuk menjelaskan 98% variansi
n_components = np.argmax(cumulative_variance >= 0.98) + 1
print(f"Jumlah komponen PCA yang menjelaskan 98% variansi: {n_components}")

# Terapkan PCA dengan jumlah komponen terbaik
pca = PCA(n_components=n_components)
train_hog_features_pca = pca.fit_transform(train_hog_features_scaled)
test_hog_features_pca = pca.transform(test_hog_features_scaled)

# Output bentuk baru dari fitur
print(f"Train Features Shape after PCA: {train_hog_features_pca.shape}")
print(f"Test Features Shape after PCA: {test_hog_features_pca.shape}")

# Validasi ukuran dataset landmark
if train_hog_features_pca.shape[0] != train_landmark_features.shape[0]:
    raise ValueError("Jumlah sampel pada PCA HOG dan landmark (train) tidak sama")
if test_hog_features_pca.shape[0] != test_landmark_features.shape[0]:
    raise ValueError("Jumlah sampel pada PCA HOG dan landmark (test) tidak sama")

# Gabungkan fitur PCA HOG dengan landmark
train_combined_features = np.hstack((train_hog_features_pca, train_landmark_features))
test_combined_features = np.hstack((test_hog_features_pca, test_landmark_features))

# Validasi label
if not np.array_equal(hog_train_labels, landmark_train_labels):
    raise ValueError("Label pada HOG dan landmark untuk training tidak sama")
if not np.array_equal(hog_test_labels, landmark_test_labels):
    raise ValueError("Label pada HOG dan landmark untuk testing tidak sama")

# Gunakan label dari salah satu (karena mereka identik)
train_labels = hog_train_labels
test_labels = hog_test_labels

# Output bentuk gabungan fitur
print(f"Train Combined Features Shape: {train_combined_features.shape}")
print(f"Test Combined Features Shape: {test_combined_features.shape}")


Jumlah komponen PCA yang menjelaskan 98% variansi: 3244
Train Features Shape after PCA: (3955, 3244)
Test Features Shape after PCA: (989, 3244)


ValueError: Jumlah sampel pada PCA HOG dan landmark (train) tidak sama

In [3]:
train_landmark_features.shape

(3983, 33)

In [4]:
train_hog_features.shape

(3955, 108900)